# Threshold Berbasis Skenario Biaya

Model memberi skor risiko pada tiap pelanggan. Threshold adalah skor minimum agar pelanggan dihubungi. Notebook ini menghitung threshold untuk tiga pilihan rasio biaya, supaya perusahaan atau pembaca repo bisa memilih rasio yang sesuai tanpa melatih ulang model.

**Model:** XGBoost, parameter default, tanpa pembobotan, 12 fitur (keputusan 31).

**Pilihan:** rasio 1:2, 1:5, dan 1:10, dengan default 1:5 (keputusan 32).

**Arti rasio:** 1:5 berarti satu churn terlewat dianggap sama merugikannya dengan lima kali salah hubungi.

**Rumus biaya:** `biaya = salah hubungi + rasio × churn terlewat`. Satuannya adalah "kali salah hubungi", bukan uang.

**Asumsi:**

- Dataset tidak punya data biaya, jadi rasio adalah asumsi.
- Biaya promo untuk pelanggan yang berisiko churn tidak dihitung, dan keberhasilan menahan pelanggan diabaikan.
- Skor belum terkalibrasi, jadi skor tidak dibaca sebagai peluang sungguhan.
- Threshold berasal dari model yang dilatih pada 80% data latih, sedangkan model akhir dilatih pada seluruh data latih.

**Keterbatasan:** Analisis mengikuti dokumentasi UCI yang menyatakan tiap baris mewakili satu pelanggan. Audit menemukan pola berulang per 50 baris, sehingga independensi antarbaris tidak terjamin dan skor model bisa terlalu optimistis. Data uji dibagi acak dan memuat pola yang sama dengan data latih, jadi hasilnya tidak boleh dibaca sebagai performa pada pelanggan baru.

## 1. Persiapan

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_score, recall_score
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from xgboost import XGBClassifier

SEED = 180915
N_SPLITS = 5
N_REPEATS = 10
N_BOOTSTRAP = 2000
RATIOS = (2, 5, 10)
DEFAULT_RATIO = 5
THRESHOLD_GRID = np.round(np.arange(0.01, 1.0, 0.01), 2)

# Path relatif terhadap root project
ROOT = Path.cwd().parent if Path.cwd().name == 'program' else Path.cwd()
OUTPUT_PATH = ROOT / 'docs' / 'threshold_skenario.csv'

In [2]:
df = pd.read_csv(ROOT / 'dataset_clean' / 'customer_churn_clean.csv')

TARGET = 'Churn'
FEATURES = [col for col in df.columns if col not in (TARGET, 'Age_Binning')]
assert TARGET not in FEATURES and 'audit_group' not in FEATURES and 'Age_Binning' not in FEATURES
assert len(FEATURES) == 12

# Pembagian yang sama dengan notebook perbandingan model dan evaluasi data uji
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=df[TARGET], random_state=SEED
)
assert not set(train_idx) & set(test_idx)
train = df.loc[train_idx].reset_index(drop=True)
test = df.loc[test_idx].reset_index(drop=True)
y_train, y_test = train[TARGET].to_numpy(), test[TARGET].to_numpy()
assert len(train) == 2280 and y_train.sum() == 357
assert len(test) == 570 and y_test.sum() == 89

print(f'Latih: {len(train):,} baris, {y_train.sum():,} churn')
print(f'Uji  : {len(test):,} baris, {y_test.sum():,} churn')

Latih: 2,280 baris, 357 churn
Uji  : 570 baris, 89 churn


## 2. Skor out-of-fold pada data latih

Tiap baris latih diberi skor oleh model yang tidak melihat baris itu. Prosesnya diulang 10 kali dengan pembagian fold yang berbeda. Data uji belum dipakai di sini. PR-AUC yang dicetak di bawah dihitung dari gabungan prediksi satu ulangan, hanya untuk memastikan skornya sama dengan notebook lain. Angka ini bukan cara pemilihan model, yang memakai skor per fold (keputusan 22), jadi tidak dibandingkan langsung dengan skor Fase 2.

In [3]:
def new_model():
    return XGBClassifier(random_state=SEED, eval_metric='logloss')


splitter = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED + 1)
splits = list(splitter.split(train, y_train))

oof_scores = np.zeros((N_REPEATS, len(train)))
for repeat in range(N_REPEATS):
    for fit_idx, val_idx in splits[repeat * N_SPLITS:(repeat + 1) * N_SPLITS]:
        model = new_model().fit(train[FEATURES].iloc[fit_idx], y_train[fit_idx])
        oof_scores[repeat, val_idx] = model.predict_proba(train[FEATURES].iloc[val_idx])[:, 1]

repeat_pr = [average_precision_score(y_train, scores) for scores in oof_scores]
print(f'PR-AUC gabungan per ulangan (cek kesamaan skor, bukan cara pemilihan model): {min(repeat_pr):.3f} sampai {max(repeat_pr):.3f}')

PR-AUC gabungan per ulangan (cek kesamaan skor, bukan cara pemilihan model): 0.923 sampai 0.938


## 3. Memilih threshold

Pada tiap ulangan, dicari threshold dengan biaya terendah. Threshold akhir adalah median dari 10 ulangan, dan rentangnya menunjukkan seberapa besar threshold bisa bergeser. Threshold F1 terbaik dipilih dengan cara yang sama, dan threshold 0,50 menjadi pembanding tetap.

In [4]:
def confusion(y_true, y_score, threshold):
    contacted = y_score >= threshold
    caught = int((contacted & (y_true == 1)).sum())
    wrong = int((contacted & (y_true == 0)).sum())
    missed = int((~contacted & (y_true == 1)).sum())
    return caught, wrong, missed


def cost(wrong, missed, ratio):
    return wrong + ratio * missed


def f1_from_counts(caught, wrong, missed):
    denominator = 2 * caught + wrong + missed
    return 2 * caught / denominator if denominator else 0.0


def best_threshold(y_true, y_score, objective):
    # Objective lebih kecil lebih baik. Kalau seri, threshold terkecil yang dipakai
    values = [objective(*confusion(y_true, y_score, th)) for th in THRESHOLD_GRID]
    return THRESHOLD_GRID[int(np.argmin(values))]


def summarize_selection(objective):
    chosen = [best_threshold(y_train, scores, objective) for scores in oof_scores]
    # Dibulatkan supaya sisa pembulatan komputer tidak ikut tersimpan
    return round(float(np.median(chosen)), 3), float(min(chosen)), float(max(chosen))


selections = {}
for ratio in RATIOS:
    selections[f'1:{ratio}'] = summarize_selection(lambda c, w, m, r=ratio: cost(w, m, r))
selections['F1 terbaik'] = summarize_selection(lambda c, w, m: -f1_from_counts(c, w, m))
selections['0,50'] = (0.5, 0.5, 0.5)

selection_table = pd.DataFrame(selections, index=['Threshold', 'Minimum', 'Maksimum']).T
print(selection_table.round(3).to_string())

scenario_thresholds = [selections[f'1:{ratio}'][0] for ratio in RATIOS]
assert scenario_thresholds == sorted(scenario_thresholds, reverse=True)

            Threshold  Minimum  Maksimum
1:2             0.230     0.12      0.35
1:5             0.060     0.05      0.14
1:10            0.045     0.02      0.08
F1 terbaik      0.300     0.19      0.43
0,50            0.500     0.50      0.50


## 4. Hasil pada data latih dengan threshold yang dipatok

Threshold median dipakai apa adanya pada semua ulangan, lalu hasilnya dirata-ratakan. Cara ini lebih jujur daripada memakai threshold terbaik tiap ulangan, tetapi threshold tetap dipilih dari prediksi yang sama, jadi hasilnya masih bisa sedikit optimistis.

In [5]:
def report_row(name, threshold, y_true, y_score_list, ratio=None):
    counts = np.array([confusion(y_true, scores, threshold) for scores in y_score_list], dtype=float)
    caught, wrong, missed = counts.mean(axis=0)
    contacted = caught + wrong
    row = {
        'Skenario': name,
        'Threshold': threshold,
        'Dihubungi': contacted,
        'Dihubungi (%)': 100 * contacted / len(y_true),
        'Tertangkap': caught,
        'Terlewat': missed,
        'Salah hubungi': wrong,
        'Precision': caught / contacted,
        'Recall': caught / (caught + missed),
        'F1': f1_from_counts(caught, wrong, missed),
        'Biaya': cost(wrong, missed, ratio) if ratio else np.nan,
    }
    return row


def scenario_rows(y_true, y_score_list):
    rows = [report_row('0,50', 0.5, y_true, y_score_list),
            report_row('F1 terbaik', selections['F1 terbaik'][0], y_true, y_score_list)]
    for ratio in RATIOS:
        name = f'1:{ratio}'
        rows.append(report_row(name, selections[name][0], y_true, y_score_list, ratio))
    return pd.DataFrame(rows).set_index('Skenario')


train_report = scenario_rows(y_train, oof_scores)
assert np.allclose(train_report['Tertangkap'] + train_report['Terlewat'], y_train.sum())
contacted_by_ratio = train_report.loc[[f'1:{ratio}' for ratio in RATIOS], 'Dihubungi']
assert contacted_by_ratio.is_monotonic_increasing
print(train_report.round(3).to_string())

            Threshold  Dihubungi  Dihubungi (%)  Tertangkap  Terlewat  Salah hubungi  Precision  Recall     F1  Biaya
Skenario                                                                                                             
0,50            0.500      336.4         14.754       293.9      63.1           42.5      0.874   0.823  0.848    NaN
F1 terbaik      0.300      372.0         16.316       311.8      45.2           60.2      0.838   0.873  0.855    NaN
1:2             0.230      386.1         16.934       317.1      39.9           69.0      0.821   0.888  0.853  148.8
1:5             0.060      470.2         20.623       337.1      19.9          133.1      0.717   0.944  0.815  232.6
1:10            0.045      490.8         21.526       339.5      17.5          151.3      0.692   0.951  0.801  326.3

## 5. Kepekaan terhadap rasio

Tabel ini menunjukkan biaya tiap threshold (baris) kalau rasio sebenarnya adalah rasio di kolom. Angka terkecil pada tiap kolom adalah threshold yang paling cocok untuk rasio itu. Dari sini terlihat dampaknya kalau rasio yang dipilih ternyata keliru.

In [6]:
sensitivity = {}
for true_ratio in RATIOS:
    column = {}
    for ratio in RATIOS:
        threshold = selections[f'1:{ratio}'][0]
        counts = np.array([confusion(y_train, scores, threshold) for scores in oof_scores], dtype=float)
        _, wrong, missed = counts.mean(axis=0)
        column[f'Threshold 1:{ratio} ({threshold:.3f})'] = cost(wrong, missed, true_ratio)
    sensitivity[f'Rasio sebenarnya 1:{true_ratio}'] = column
print(pd.DataFrame(sensitivity).round(1).to_string())

                        Rasio sebenarnya 1:2  Rasio sebenarnya 1:5  Rasio sebenarnya 1:10
Threshold 1:2 (0.230)                  148.8                 268.5                  468.0
Threshold 1:5 (0.060)                  172.9                 232.6                  332.1
Threshold 1:10 (0.045)                 186.3                 238.8                  326.3


## 6. Penjelasan tiap rasio

| Rasio | Artinya | Biasanya cocok jika | Contoh kondisi (ilustrasi) |
|---|---|---|---|
| **1:2** | Churn terlewat 2 kali lebih merugikan daripada salah hubungi | Biaya menghubungi cukup mahal, sehingga salah sasaran perlu ditekan | Diskon besar atau telepon oleh petugas |
| **1:5** (default) | Churn terlewat 5 kali lebih merugikan daripada salah hubungi | Biaya menghubungi sedang, dan kedua kesalahan sama-sama diperhatikan | Promo sedang lewat telepon atau aplikasi |
| **1:10** | Churn terlewat 10 kali lebih merugikan daripada salah hubungi | Biaya menghubungi murah, sedangkan kehilangan pelanggan sangat merugikan | SMS murah untuk pelanggan bernilai tinggi |

**Cara menghitung rasio sendiri:** `rasio = kerugian per churn terlewat ÷ biaya per salah hubungi`. Misalnya, kerugian Rp100.000 dan biaya Rp20.000 menghasilkan 5, jadi pilih 1:5. Angka rupiah ini hanya ilustrasi, bukan dari dataset. Kalau hasilnya di antara pilihan, pilih yang paling dekat.

Default 1:5 adalah titik awal, bukan rasio yang terbukti benar.

## 7. Menyimpan threshold

Threshold disimpan sebelum data uji dipakai. File ini nanti dibaca demo di Fase 5.

In [7]:
EXPLANATIONS = {
    2: 'Churn terlewat dianggap 2 kali lebih merugikan daripada salah hubungi. Cocok jika biaya menghubungi cukup mahal.',
    5: 'Churn terlewat dianggap 5 kali lebih merugikan daripada salah hubungi. Cocok jika biaya menghubungi sedang.',
    10: 'Churn terlewat dianggap 10 kali lebih merugikan daripada salah hubungi. Cocok jika biaya menghubungi murah.',
}

threshold_table = pd.DataFrame([
    {
        'rasio': f'1:{ratio}',
        'threshold': selections[f'1:{ratio}'][0],
        'threshold_min': selections[f'1:{ratio}'][1],
        'threshold_max': selections[f'1:{ratio}'][2],
        'is_default': ratio == DEFAULT_RATIO,
        'penjelasan': EXPLANATIONS[ratio],
    }
    for ratio in RATIOS
])
assert len(threshold_table) == 3 and threshold_table['is_default'].sum() == 1
OUTPUT_PATH.parent.mkdir(exist_ok=True)
# Akhir baris dipatok supaya isi file sama di semua sistem operasi
threshold_table.to_csv(OUTPUT_PATH, index=False, lineterminator='\n')
print(threshold_table.drop(columns='penjelasan').to_string(index=False))

rasio  threshold  threshold_min  threshold_max  is_default
  1:2      0.230           0.12           0.35       False
  1:5      0.060           0.05           0.14        True
 1:10      0.045           0.02           0.08       False


## 8. Laporan pada data uji

Model dilatih pada seluruh data latih, lalu menilai data uji memakai threshold yang sudah disimpan. Hasil ini hanya laporan, bukan dasar memilih threshold atau rasio.

Rentang memakai bootstrap persentil 95% pada baris uji. Rentang ini mengasumsikan baris independen dan threshold tetap, jadi bisa terlalu sempit.

In [8]:
final_model = new_model().fit(train[FEATURES], y_train)
test_scores = final_model.predict_proba(test[FEATURES])[:, 1]

test_report = scenario_rows(y_test, [test_scores])
assert (test_report['Tertangkap'] + test_report['Terlewat'] == y_test.sum()).all()


def bootstrap_interval(y_true, y_pred, metric, seed=SEED, n_boot=N_BOOTSTRAP):
    rng = np.random.default_rng(seed)
    values = []
    while len(values) < n_boot:
        idx = rng.integers(0, len(y_true), len(y_true))
        # Sampel tanpa churn tidak bisa dinilai
        if y_true[idx].max() == 0:
            continue
        values.append(metric(y_true[idx], y_pred[idx], zero_division=0))
    return np.percentile(values, [2.5, 97.5])


def fmt(value, interval):
    return f'{value:.3f} ({interval[0]:.3f}–{interval[1]:.3f})'.replace('.', ',')


rows = []
for name, row in test_report.iterrows():
    predicted = (test_scores >= row['Threshold']).astype(int)
    rows.append({
        'Skenario': name,
        'Precision': fmt(precision_score(y_test, predicted), bootstrap_interval(y_test, predicted, precision_score)),
        'Recall': fmt(recall_score(y_test, predicted), bootstrap_interval(y_test, predicted, recall_score)),
    })

print(test_report.round(3).to_string())
print()
print(pd.DataFrame(rows).set_index('Skenario').to_string())

            Threshold  Dihubungi  Dihubungi (%)  Tertangkap  Terlewat  Salah hubungi  Precision  Recall     F1  Biaya
Skenario                                                                                                             
0,50            0.500       73.0         12.807        67.0      22.0            6.0      0.918   0.753  0.827    NaN
F1 terbaik      0.300       87.0         15.263        78.0      11.0            9.0      0.897   0.876  0.886    NaN
1:2             0.230       95.0         16.667        81.0       8.0           14.0      0.853   0.910  0.880   30.0
1:5             0.060      117.0         20.526        87.0       2.0           30.0      0.744   0.978  0.845   40.0
1:10            0.045      125.0         21.930        87.0       2.0           38.0      0.696   0.978  0.813   58.0

                      Precision               Recall
Skenario                                            
0,50        0,918 (0,851–0,973)  0,753 (0,662–0,838)
F1 terbaik  0,

## Cara membaca hasil

- Rasio adalah asumsi. Tabel menunjukkan akibat tiap asumsi, bukan rasio yang benar untuk suatu perusahaan.
- Biaya pada tabel adalah hitungan dengan satuan "kali salah hubungi", bukan penghematan uang. Biaya antar-rasio tidak bisa dibandingkan langsung, karena satuannya berbeda.
- Threshold bisa bergeser, seperti terlihat pada rentangnya, dan skor belum terkalibrasi.
- Pelanggan yang skornya di atas threshold berhubungan dengan pola churn pada data ini. Hasil ini tidak membuktikan bahwa menghubungi mereka akan mencegah churn.
- Skor CV acak dan data uji tidak boleh disebut performa pada pelanggan baru.